# تحليل أداء طلبات الوجبات

يجيب هذا الدفتر عن أسئلة الطلب الأسبوعي، والفئات والوجبات والمراكز، والترويج والخصم، وتذبذب الطلب. تُقرأ ملفات المصدر من `data/raw/` فقط؛ لا تكتب أي مخرجات أو تعديلات إلى هذا المجلد. ملف `test.csv` مخصص للتنبؤ الاختياري ولا يحتوي على `num_orders`، لذلك لا يدخل في التحليل الوصفي أو الاختبارات أدناه.

التحليل وصفي ورصدي: اختبارات Mann–Whitney وSpearman لا تثبت السببية. فسّر النتائج مع مراعاة اختلاف الوجبات والمراكز والتكرار والقيود الموضحة في نهاية الدفتر.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'raw' / 'train.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find data/raw/train.csv from the notebook working directory.')

ROOT = find_project_root(Path.cwd())
RAW = ROOT / 'data' / 'raw'
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

def audit(frame: pd.DataFrame, label: str) -> None:
    if 'num_orders' in frame:
        order_total = f" | num_orders sum: {frame['num_orders'].sum():,.0f}"
    elif 'total_orders' in frame:
        order_total = f" | total_orders sum: {frame['total_orders'].sum():,.0f}"
    else:
        order_total = ''
    print(f'{label}: rows={len(frame):,}, columns={frame.shape[1]}{order_total}')

train = pd.read_csv(RAW / 'train.csv')
centers = pd.read_csv(RAW / 'fulfilment_center_info.csv')
meals = pd.read_csv(RAW / 'meal_info.csv')
test_path = RAW / 'test.csv'
test = pd.read_csv(test_path) if test_path.exists() else None

for name, frame in [('train.csv', train), ('fulfilment_center_info.csv', centers), ('meal_info.csv', meals)]:
    audit(frame, name)
    print('columns:', frame.columns.tolist())
    missing = frame.isna().sum()
    print('missing values:', missing[missing.gt(0)].to_dict() or 'none')
if test is not None:
    audit(test, 'test.csv (reserved; no target column)')

print('Duplicate train ids:', train['id'].duplicated().sum())
print('Duplicate train (week, center_id, meal_id) keys:', train.duplicated(['week', 'center_id', 'meal_id']).sum())
print('Duplicate center_id lookup keys:', centers['center_id'].duplicated().sum())
print('Duplicate meal_id lookup keys:', meals['meal_id'].duplicated().sum())

train.csv: rows=456,548, columns=9 | num_orders sum: 119,557,485
columns: ['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders']
missing values: none
fulfilment_center_info.csv: rows=77, columns=5
columns: ['center_id', 'city_code', 'region_code', 'center_type', 'op_area']
missing values: none
meal_info.csv: rows=51, columns=3
columns: ['meal_id', 'category', 'cuisine']
missing values: none
test.csv (reserved; no target column): rows=32,573, columns=8
Duplicate train ids: 0
Duplicate train (week, center_id, meal_id) keys: 0
Duplicate center_id lookup keys: 0
Duplicate meal_id lookup keys: 0


## ربط البيانات والتحقق

نستخدم ربطاً يسارياً مع التحقق من أن مفاتيح الجداول الوصفية فريدة (`many_to_one`). تُطبع أعداد الصفوف والأعمدة ومجموع الطلبات قبل كل ربط وبعده؛ أي تغيّر في عدد السجلات أو مجموع الطلبات يحتاج إلى تفسير قبل متابعة التحليل.

In [2]:
orders = train.copy()
audit(orders, 'Before center join')
orders = orders.merge(centers, on='center_id', how='left', validate='many_to_one', indicator='_center_merge')
audit(orders, 'After center join')
print('Unmatched centers:', int(orders['_center_merge'].eq('left_only').sum()))
orders = orders.drop(columns='_center_merge')
audit(orders, 'After removing center-join diagnostic column')

audit(orders, 'Before meal join')
orders = orders.merge(meals, on='meal_id', how='left', validate='many_to_one', indicator='_meal_merge')
audit(orders, 'After meal join')
print('Unmatched meals:', int(orders['_meal_merge'].eq('left_only').sum()))
orders = orders.drop(columns='_meal_merge')
audit(orders, 'After removing meal-join diagnostic column')

assert len(orders) == len(train), 'A lookup join changed the number of order rows.'
assert orders['num_orders'].sum() == train['num_orders'].sum(), 'A lookup join changed total orders.'
assert not orders[['center_type', 'op_area', 'category', 'cuisine']].isna().any().any(), 'Unmatched lookup data requires investigation.'
print('Join validation passed: row count and order total are unchanged.')

Before center join: rows=456,548, columns=9 | num_orders sum: 119,557,485
After center join: rows=456,548, columns=14 | num_orders sum: 119,557,485
Unmatched centers: 0
After removing center-join diagnostic column: rows=456,548, columns=13 | num_orders sum: 119,557,485
Before meal join: rows=456,548, columns=13 | num_orders sum: 119,557,485
After meal join: rows=456,548, columns=16 | num_orders sum: 119,557,485
Unmatched meals: 0
After removing meal-join diagnostic column: rows=456,548, columns=15 | num_orders sum: 119,557,485
Join validation passed: row count and order total are unchanged.


## فحص الجودة وقرارات التنظيف

نقيس المشكلات المحتملة أولاً قبل حذف أي سجل. لا يُحذف أي صف لمجرد كونه متطرفاً؛ فالطلب المرتفع قد يكون طلباً حقيقياً. إذا أضيف تنظيف لاحقاً، يجب توثيق السبب وطباعة الصفوف والأعمدة ومجموع الطلبات قبل العملية وبعدها.

In [3]:
audit(orders, 'Joined data before quality review')
print('Missing values by column:', orders.isna().sum()[orders.isna().sum().gt(0)].to_dict() or 'none')
print('Duplicate business keys (week, center_id, meal_id):', int(orders.duplicated(['week', 'center_id', 'meal_id']).sum()))
print('Rows with checkout_price > base_price:', int(orders['checkout_price'].gt(orders['base_price']).sum()))
print('Rows with non-positive prices:', int((orders[['checkout_price', 'base_price']] <= 0).any(axis=1).sum()))
print('Rows with negative orders:', int(orders['num_orders'].lt(0).sum()))

q1, q3 = orders['num_orders'].quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = orders['num_orders'].lt(q1 - 1.5 * iqr) | orders['num_orders'].gt(q3 + 1.5 * iqr)
print('IQR-flagged order rows (review, not automatically removed):', int(outlier_mask.sum()))

all_weeks = set(range(int(orders['week'].min()), int(orders['week'].max()) + 1))
week_coverage = orders.groupby(['center_id', 'meal_id'])['week'].agg(lambda values: len(all_weeks - set(values)))
print('Center-meal series with missing weeks:', int(week_coverage.gt(0).sum()))
print('Total missing center-meal weeks:', int(week_coverage.sum()))

# No records are dropped at this stage: anomalies are retained until investigated.
analysis = orders.copy()
audit(analysis, 'Analysis data after documented no-drop cleaning decision')

Joined data before quality review: rows=456,548, columns=15 | num_orders sum: 119,557,485
Missing values by column: none
Duplicate business keys (week, center_id, meal_id): 0
Rows with checkout_price > base_price: 116101
Rows with non-positive prices: 0
Rows with negative orders: 0
IQR-flagged order rows (review, not automatically removed): 32937
Center-meal series with missing weeks: 2350
Total missing center-meal weeks: 65017
Analysis data after documented no-drop cleaning decision: rows=456,548, columns=15 | num_orders sum: 119,557,485


## مؤشرات الطلب والتذبذب

الإيراد التقديري هنا يساوي سعر الدفع مضروباً في الطلبات المسجلة. الخصم النسبي يُحسب مقابل `base_price` عندما يكون السعر الأساسي موجباً. أرقام هذا القسم تُستخرج مباشرة من الجداول المعروضة ويمكن الاستشهاد بها في التقرير.

In [4]:
analysis = analysis.assign(
    estimated_revenue=analysis['checkout_price'] * analysis['num_orders'],
    discount_pct=np.where(analysis['base_price'] > 0, (analysis['base_price'] - analysis['checkout_price']) / analysis['base_price'] * 100, np.nan),
)
print(f"Total orders: {analysis['num_orders'].sum():,.0f}")
print(f"Mean weekly orders: {analysis.groupby('week')['num_orders'].sum().mean():,.2f}")
print(f"Estimated revenue: {analysis['estimated_revenue'].sum():,.2f}")
print(f"Rows with email promotion: {analysis['emailer_for_promotion'].mean():.2%}")
print(f"Rows with homepage feature: {analysis['homepage_featured'].mean():.2%}")

weekly = analysis.groupby('week', as_index=False)['num_orders'].sum()
category_demand = analysis.groupby('category', as_index=False)['num_orders'].sum().sort_values('num_orders', ascending=False)
meal_demand = analysis.groupby(['meal_id', 'category', 'cuisine'], as_index=False)['num_orders'].sum().sort_values('num_orders', ascending=False)
meal_demand['cumulative_share'] = meal_demand['num_orders'].cumsum() / meal_demand['num_orders'].sum()
top_80_meals = meal_demand.loc[meal_demand['cumulative_share'].shift(fill_value=0) < 0.8].copy()
center_week = analysis.groupby(['center_type', 'center_id', 'week'], as_index=False)['num_orders'].sum()
center_week_average = center_week.groupby(['center_type', 'center_id'], as_index=False)['num_orders'].mean().rename(columns={'num_orders': 'mean_weekly_orders_per_center'})
center_type_demand = center_week_average.groupby('center_type', as_index=False).agg(mean_weekly_orders_per_center=('mean_weekly_orders_per_center', 'mean'), median_weekly_orders_per_center=('mean_weekly_orders_per_center', 'median'), centers=('center_id', 'nunique'))
center_type_total = analysis.groupby('center_type', as_index=False)['num_orders'].sum().rename(columns={'num_orders': 'total_orders'})
audit(center_type_demand, 'Center-type summary before total-orders join')
center_type_demand = center_type_demand.merge(center_type_total, on='center_type', how='left', validate='one_to_one')
audit(center_type_demand, 'Center-type summary after total-orders join')

meal_weekly = analysis.groupby(['center_id', 'meal_id', 'week'], as_index=False)['num_orders'].sum()
variability = meal_weekly.groupby(['center_id', 'meal_id'])['num_orders'].agg(mean='mean', std='std', series_total_orders='sum').reset_index()
variability['cv'] = variability['std'] / variability['mean'].replace(0, np.nan)

display(weekly.head(), category_demand, top_80_meals, center_type_demand, variability.sort_values('cv', ascending=False).head(15))

Total orders: 119,557,485
Mean weekly orders: 824,534.38
Estimated revenue: 31,922,771,040.11
Rows with email promotion: 8.12%
Rows with homepage feature: 10.92%
 week  num_orders
    1      792261
    2      787084
    3      695262
    4      743529
    5     1198675
    category  num_orders
   Beverages    40480525
   Rice Bowl    20874063
    Sandwich    17636782
       Salad    10944336
       Pizza     7383720
Other Snacks     4766293
    Starters     4649122
      Extras     3984979
     Seafood     2715714
      Desert     1940754
       Pasta     1637744
        Soup     1039646
        Fish      871959
     Biryani      631848
 meal_id  category     cuisine  num_orders  cumulative_share
    2290 Rice Bowl      Indian     9770303          0.081721
    1885 Beverages        Thai     8346246          0.151530
    1754  Sandwich     Italian     6818059          0.208557
    1993 Beverages        Thai     6656856          0.264237
    1109 Rice Bowl      Indian     6367773        

## الاختبارات الإحصائية

يقارن اختبار Mann–Whitney الطلب مع الترويج وبدونه داخل كل مجموعة وجبة ومركز تتوفر فيها مشاهدتان على الأقل في كل حالة. تُعرض النتائج حسب المجموعة؛ تكرار الاختبارات يعني أن قيم p غير مصححة لتعدد المقارنات، لذا تُعامل كاستكشافية. اختبار Spearman يصف الارتباط الرتبي بين الخصم والطلب على مستوى الصفوف، ولا يعزل أثر بقية العوامل ولا يثبت السببية.

In [5]:
try:
    from scipy.stats import mannwhitneyu, spearmanr
except ImportError as exc:
    raise ImportError('Install scipy in the selected notebook environment to run the required statistical tests.') from exc

promotion_results = []
for flag in ['emailer_for_promotion', 'homepage_featured']:
    for (center_id, meal_id), group in analysis.groupby(['center_id', 'meal_id'], sort=False):
        no_promo = group.loc[group[flag].eq(0), 'num_orders']
        promo = group.loc[group[flag].eq(1), 'num_orders']
        if len(no_promo) >= 2 and len(promo) >= 2:
            test_result = mannwhitneyu(promo, no_promo, alternative='two-sided')
            baseline = no_promo.median()
            median_change_pct = ((promo.median() - baseline) / baseline * 100) if baseline else np.nan
            promotion_results.append({'promotion_flag': flag, 'center_id': center_id, 'meal_id': meal_id, 'n_promo': len(promo), 'n_no_promo': len(no_promo), 'median_orders_promo': promo.median(), 'median_orders_no_promo': baseline, 'median_change_pct': median_change_pct, 'u_statistic': test_result.statistic, 'p_value': test_result.pvalue})

promotion_tests = pd.DataFrame(promotion_results)
print('Eligible promotion-by-center-meal strata:', len(promotion_tests))
if not promotion_tests.empty:
    promotion_summary = promotion_tests.groupby('promotion_flag').agg(strata=('p_value', 'size'), median_within_stratum_change_pct=('median_change_pct', 'median'), unadjusted_p_below_005=('p_value', lambda values: int(values.lt(0.05).sum()))).reset_index()
    display(promotion_summary)
    display(promotion_tests.sort_values('p_value').groupby('promotion_flag').head(10))

spearman_data = analysis[['discount_pct', 'num_orders']].copy()
audit(spearman_data, 'Spearman rows before invalid-value cleanup')
spearman_data = spearman_data.replace([np.inf, -np.inf], np.nan)
audit(spearman_data, 'Spearman rows after replacing infinite values')
spearman_data = spearman_data.dropna()
audit(spearman_data, 'Spearman rows after dropping missing values')
spearman_result = spearmanr(spearman_data['discount_pct'], spearman_data['num_orders'])
p_display = f"{spearman_result.pvalue:.4g}" if spearman_result.pvalue > 0 else '<1e-300 (numerical underflow)'
print(f"Spearman rho={spearman_result.statistic:.4f}, p-value={p_display}, n={len(spearman_data):,}")

Eligible promotion-by-center-meal strata: 5612
       promotion_flag  strata  median_within_stratum_change_pct  unadjusted_p_below_005
emailer_for_promotion    2814                        151.858762                    2380
    homepage_featured    2798                        134.683236                    2350
       promotion_flag  center_id  meal_id  n_promo  n_no_promo  median_orders_promo  median_orders_no_promo  median_change_pct  u_statistic      p_value
    homepage_featured         30     1993       56          89               1323.5                   649.0         103.929122       4929.0 4.360215e-23
    homepage_featured         23     1962       38         107                642.0                   136.0         372.058824       3978.0 2.259054e-18
emailer_for_promotion         23     1962       40         105                634.0                   136.0         366.176471       4066.5 3.376168e-18
emailer_for_promotion         99     1962       40         105               

## الرسوم

الرسوم التالية محفوظة في `reports/figures/` ويمكن ربطها بنتائج الدفتر عند إعداد التقرير. عناوينها تصف المتغيرات المعروضة دون ادعاء علاقة سببية.

In [6]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekly['week'], weekly['num_orders'])
ax.set(title='إجمالي الطلبات يتغير بين الأسابيع المسجلة', xlabel='الأسبوع', ylabel='عدد الطلبات')
fig.tight_layout(); fig.savefig(FIGURES / 'weekly_demand.png', dpi=160); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(category_demand['category'].astype(str), category_demand['num_orders'])
ax.set(title='المشروبات أعلى الفئات في إجمالي الطلبات', xlabel='الفئة', ylabel='عدد الطلبات')
ax.tick_params(axis='x', rotation=45); fig.tight_layout(); fig.savefig(FIGURES / 'category_demand.png', dpi=160); plt.show(); plt.close(fig)

for flag, title, filename in [('emailer_for_promotion', 'البريد الترويجي', 'email_promotion_orders_boxplot.png'), ('homepage_featured', 'الظهور في الصفحة الرئيسية', 'homepage_featured_orders_boxplot.png')]:
    fig, ax = plt.subplots(figsize=(7, 4))
    promotion_groups = [analysis.loc[analysis[flag].eq(value), 'num_orders'] for value in [0, 1]]
    ax.boxplot(promotion_groups, tick_labels=[f'بدون {title}', f'مع {title}'], showfliers=False)
    ax.set(title=f'توزيع الطلبات يختلف حسب {title}', ylabel='عدد الطلبات')
    fig.tight_layout(); fig.savefig(FIGURES / filename, dpi=160); plt.show(); plt.close(fig)

scatter_data = analysis.sample(min(10000, len(analysis)), random_state=42)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(scatter_data['discount_pct'], scatter_data['num_orders'], alpha=0.25, s=10)
ax.set(title=f'ارتباط الخصم بالطلبات ضعيف وموجب (Spearman rho={spearman_result.statistic:.3f})', xlabel='الخصم النسبي (%)', ylabel='عدد الطلبات')
fig.tight_layout(); fig.savefig(FIGURES / 'discount_vs_orders.png', dpi=160); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(variability['series_total_orders'], variability['cv'], alpha=0.5, s=16)
ax.set(title='تذبذب الطلب يختلف بين سلاسل المركز والوجبة', xlabel='إجمالي طلبات الوجبة في المركز', ylabel='معامل التذبذب (CV)')
fig.tight_layout(); fig.savefig(FIGURES / 'demand_volume_vs_variability.png', dpi=160); plt.show(); plt.close(fig)

print('Saved figures:', sorted(path.name for path in FIGURES.glob('*.png')))

Saved figures: ['category_demand.png', 'demand_volume_vs_variability.png', 'discount_vs_orders.png', 'email_promotion_orders_boxplot.png', 'homepage_featured_orders_boxplot.png', 'weekly_demand.png']


## حدود التحليل

- البيانات رصدية؛ لا تثبت أن العروض أو الخصومات سببت تغير الطلب.
- قيم الطلبات المفقودة أو الأسابيع غير المسجلة قد تؤثر في المتوسطات والتذبذب؛ لم تُستبدل الأسابيع المفقودة بصفر.
- اختبار Mann–Whitney منفذ داخل مجموعات المركز والوجبة، لكن المشاهدات الزمنية المتكررة قد لا تكون مستقلة، والاختبارات المتعددة غير مصححة.
- ارتباط Spearman المحسوب على الصفوف لا يضبط المركز أو الوجبة أو الأسبوع أو الترويج.
- الإيراد تقديري وفق السعر المسجل والطلبات، ولا يتضمن التكاليف أو الهدر أو المخزون الفعلي.
- فحص IQR يحدد قيماً تستحق المراجعة ولا يثبت أنها أخطاء.